In [ ]:
import folium
import pandas as pd

In [ ]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster, MousePosition
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

In [ ]:
# Download and read the `spacex_launch_geo.csv`
#from js import fetch
#import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
#resp = await fetch(URL)
#spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(URL)

In [ ]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [ ]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [ ]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

In [ ]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label

# التكرار عبر كل صف في جدول مواقع الإطلاق
for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]

    # 1. إنشاء وإضافة دائرة (Circle) لكل موقع
    circle = folium.Circle(
        coordinate,
        radius=50,
        color='#d35400',
        fill=True
    ).add_child(folium.Popup(row['Launch Site']))

    # 2. إنشاء وإضافة نص (Label) باستخدام DivIcon
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site'],
        )
    )

    # إضافة العناصر إلى الخريطة
    site_map.add_child(circle)
    site_map.add_child(marker)

# عرض الخريطة
site_map

In [ ]:
# Task 2: Mark the success/failed launches for each site on the map
from folium.plugins import MarkerCluster

# 1. إنشاء MarkerCluster
marker_cluster = MarkerCluster().add_to(site_map)

# 2. التكرار عبر الـ DataFrame وإضافة العلامات
for index, row in spacex_df.iterrows():
    # تحديد لون العلامة: أخضر للنجاح، أحمر للفشل
    marker_color = 'green' if row['class'] == 1 else 'red'

    # إنشاء العلامة
    marker = folium.Marker(
        location=[row['Lat'], row['Long']],
        popup=f"Launch Site: {row['Launch Site']}<br>Result: {'Success' if row['class'] == 1 else 'Failed'}",
        icon=folium.Icon(color=marker_cluster, icon_color=marker_color)
    )

    # إضافة العلامة إلى مجموعة العناقيد
    marker.add_to(marker_cluster)

# عرض الخريطة المحدثة
site_map

/tmp/ipykernel_2496/83281570.py:16: UserWarning: color argument of Icon should be one of: {'blue', 'darkblue', 'pink', 'lightgray', 'lightgreen', 'green', 'darkred', 'lightblue', 'lightred', 'gray', 'purple', 'cadetblue', 'darkgreen', 'darkpurple', 'orange', 'white', 'beige', 'red', 'black'}.
  icon=folium.Icon(color=marker_cluster, icon_color=marker_color)


In [ ]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [ ]:
marker_cluster = MarkerCluster()


In [ ]:
#TODO: Create a new column in spacex_df dataframe called marker_color to store the marker colors based on the class value

# 1. تعريف دالة لتحديد اللون بناءً على قيمة class
def get_marker_color(launch_class):
    if launch_class == 1:
        return 'green'  # أخضر للنجاح
    else:
        return 'red'    # أحمر للفشل

# 2. إنشاء العمود الجديد marker_color وتطبيق الدالة على كل صف
spacex_df['marker_color'] = spacex_df['class'].apply(get_marker_color)

# عرض آخر 10 صفوف للتأكد من إضافة العمود الجديد
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


In [ ]:
# إنشاء كائن الـ Cluster
marker_cluster = MarkerCluster()

# إضافة marker_cluster إلى الخريطة
site_map.add_child(marker_cluster)

# حلقة التكرار لإضافة كل عملية إطلاق كـ Marker داخل الـ Cluster
for index, record in spacex_df.iterrows():
    # إنشاء Marker لكل موقع إطلاق
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        # استخدام الرمز الملون الذي حددناه في العمود marker_color
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=f"Launch Site: {record['Launch Site']}<br>Result: {'Success' if record['class'] == 1 else 'Failed'}"
    )

    # إضافة الماركر إلى الـ Cluster
    marker_cluster.add_child(marker)

# عرض الخريطة
site_map

In [ ]:
from geopy.distance import geodesic

# إحداثيات موقع الإطلاق (مثال)
launch_site_coords = (28.562302, -80.577356)

# إحداثيات المعلم القريب (مثال: نقطة على الساحل)
coastline_coords = (28.563, -80.568)

# حساب المسافة
distance = geodesic(launch_site_coords, coastline_coords).km
print(f"المسافة إلى الساحل هي: {distance:.2f} كم")

المسافة إلى الساحل هي: 0.92 كم


In [ ]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

In [ ]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [ ]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# 1. إحداثيات موقع الإطلاق (كمثال لـ KSC LC-39A)
launch_site_lat = 28.573255
launch_site_lon = -80.646895

# 2. إحداثيات أقرب نقطة على الساحل (محددة بواسطة MousePosition)
coastline_lat = 28.57465
coastline_lon = -80.57075

# 3. حساب المسافة باستخدام الدالة التي عرفتها سابقاً
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# 4. طباعة النتيجة
print(f"المسافة إلى الساحل: {distance_coastline:.2f} كم")

# 5. إضافة المسافة كعلامة (Marker) على الخريطة
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:.2f} KM".format(distance_coastline),
    )
)
site_map.add_child(distance_marker)

# 6. إضافة خط (PolyLine) يربط بين موقع الإطلاق والساحل
polyline = folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]],
    weight=1
)
site_map.add_child(polyline)

# عرض الخريطة
site_map

المسافة إلى الساحل: 7.44 كم


In [ ]:
# Create and add a folium.Marker on your selected closest coastline point on the map
launch_site_lat = 28.573255
launch_site_lon = -80.646895
coastline_lat = 28.57465
coastline_lon = -80.57075
coastline_coord = [coastline_lat, coastline_lon]

# Display the distance between coastline point and launch site using the icon property
# for example
# distance_marker = folium.Marker(
#    coordinate,
#    icon=DivIcon(
#        icon_size=(20,20),
#        icon_anchor=(0,0),
#        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
#        )
#    )

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

distance_marker = folium.Marker(
    coastline_coord,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
    )
)

site_map.add_child(distance_marker)

polyline = folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], coastline_coord],
    weight=1
)
site_map.add_child(polyline)
site_map

In [ ]:
# 1. تحديد قائمة الإحداثيات [موقع الإطلاق، نقطة الساحل]
# تأكد من استبدال الإحداثيات بالقيم التي اخترتها
coordinates = [[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]]

# 2. إنشاء كائن PolyLine
lines = folium.PolyLine(locations=coordinates, weight=1)

# 3. إضافة الخط إلى الخريطة
site_map.add_child(lines)

# عرض الخريطة
site_map

In [ ]:
# قائمة بالمعالم التي تريد قياس المسافة إليها (أضف الإحداثيات التي استخرجتها بالماوس)
proximities = [
    {"name": "Coastline", "coords": [28.57465, -80.57075]},
    {"name": "Railway", "coords": [28.57210, -80.58520]},
    {"name": "Highway", "coords": [28.57350, -80.65500]}
]

# حلقة تكرار لكل معلم
for item in proximities:
    # 1. حساب المسافة
    distance = calculate_distance(launch_site_lat, launch_site_lon, item["coords"][0], item["coords"][1])

    # 2. إنشاء العلامة (Marker) مع المسافة
    distance_marker = folium.Marker(
        item["coords"],
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
        )
    )
    site_map.add_child(distance_marker)

    # 3. رسم الخط (PolyLine) بين موقع الإطلاق والمعلم
    lines = folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], item["coords"]],
        weight=1
    )
    site_map.add_child(lines)

site_map